In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

import joblib


In [2]:
df = pd.read_csv("../data/cleaned_retail.csv")

# Ensure datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Day,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,8


In [3]:
le_product = LabelEncoder()

df["ProductCode"] = le_product.fit_transform(df["Description"])

df[["Description", "ProductCode"]].head()


,Description,ProductCode
0,WHITE HANGING HEART T-LIGHT HOLDER,3698
1,WHITE METAL LANTERN,3706
2,CREAM CUPID HEARTS COAT HANGER,858
3,KNITTED UNION FLAG HOT WATER BOTTLE,1804
4,RED WOOLLY HOTTIE WHITE HEART.,2763


In [4]:
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (df["InvoiceDate"].max() - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalPrice": "sum"
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,325,1,77183.60
1,12347.0,1,7,4310.00
2,12348.0,74,4,1797.24
3,12349.0,18,1,1757.55
4,12350.0,309,1,334.40


In [5]:
last_product = (
    df.sort_values("InvoiceDate")
      .groupby("CustomerID")["ProductCode"]
      .last()
      .reset_index()
)

last_product.columns = ["CustomerID", "LastProduct"]

last_product.head()


,CustomerID,LastProduct
0,12346.0,1992
1,12347.0,790
2,12348.0,2611
3,12349.0,3151
4,12350.0,728


In [6]:
nba_df = rfm.merge(last_product, on="CustomerID", how="inner")

nba_df.head()


,CustomerID,Recency,Frequency,Monetary,LastProduct
0,12346.0,325,1,77183.60,1992
1,12347.0,1,7,4310.00,790
2,12348.0,74,4,1797.24,2611
3,12349.0,18,1,1757.55,3151
4,12350.0,309,1,334.40,728


In [7]:
next_product = (
    df.sort_values("InvoiceDate")
      .groupby("CustomerID")["ProductCode"]
      .last()
      .reset_index()
)

next_product.columns = ["CustomerID", "NextProduct"]

nba_df = nba_df.merge(next_product, on="CustomerID", how="inner")

nba_df.head()


,CustomerID,Recency,Frequency,Monetary,LastProduct,NextProduct
0,12346.0,325,1,77183.60,1992,1992
1,12347.0,1,7,4310.00,790,790
2,12348.0,74,4,1797.24,2611,2611
3,12349.0,18,1,1757.55,3151,3151
4,12350.0,309,1,334.40,728,728


In [8]:
MIN_PRODUCT_FREQ = 10   # safe for 58K rows

product_counts = nba_df["NextProduct"].value_counts()

valid_products = product_counts[product_counts >= MIN_PRODUCT_FREQ].index

nba_df = nba_df[nba_df["NextProduct"].isin(valid_products)]

print("Remaining customers:", nba_df.shape[0])
print("Remaining unique products:", nba_df["NextProduct"].nunique())


Remaining customers: 666
Remaining unique products: 47


In [9]:
le_target = LabelEncoder()

nba_df["NextProductEncoded"] = le_target.fit_transform(
    nba_df["NextProduct"]
)


In [10]:
X = nba_df[["Recency", "Frequency", "Monetary", "LastProduct"]]
y = nba_df["NextProductEncoded"]

X.head(), y.head()


(    Recency  Frequency  Monetary  LastProduct
 2        74          4   1797.24         2611
 8       213          1    459.40         2767
 13       51          3   2662.06         2996
 16      109          2    552.00         3559
 18      290          2    641.38         2130,
 2     29
 8     32
 13    37
 16    42
 18    24
 Name: NextProductEncoded, dtype: int64)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)


In [12]:
xgb_nba = XGBClassifier(
    n_estimators=80,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",   # MUCH faster than softprob
    eval_metric="mlogloss",
    tree_method="hist",          # 🔥 speed boost
    n_jobs=-1,
    random_state=42
)

xgb_nba.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [13]:
y_pred = xgb_nba.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"NBA Model Accuracy: {acc:.4f}")


NBA Model Accuracy: 1.0000


In [14]:
joblib.dump(xgb_nba, "../models/nba_model.pkl")
joblib.dump(le_product, "../models/product_encoder.pkl")
joblib.dump(le_target, "../models/target_encoder.pkl")

print("✅ NBA model & encoders saved")


✅ NBA model & encoders saved


In [15]:
nba_df["PredictedProductEncoded"] = xgb_nba.predict(X)

nba_df["PredictedProductName"] = le_product.inverse_transform(
    le_target.inverse_transform(nba_df["PredictedProductEncoded"])
)

nba_df[["CustomerID", "PredictedProductName"]].to_csv(
    "../data/predicted_nba.csv",
    index=False
)

print("✅ Predicted NBA saved for frontend/dashboard")


✅ Predicted NBA saved for frontend/dashboard
